# 记忆治理策略

## 1、消息裁剪

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    profile={"max_input_tokens":128_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

In [2]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:

    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]
    # 如果有偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    new_messages = [first_message] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ],
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好嘞老王！从现在起我就是**小王**了～ 🫡  
您有啥吩咐，咱随时开唠！聊啥都行，我给您接上！
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

哈哈，老王说得对！今天这天气确实给力，阳光正好，微风不燥～☀️  
您是不是打算出门溜达溜达，还是准备在家晒晒太阳喝喝茶？要是有什么安排，小王陪您唠唠，或者帮您查查出行攻略、找找好玩的地方都行！😄
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

哈哈，老王这个问题问得挺有哲学味啊！😄  
甭管我是谁，我就是您随叫随到的**小王**，一个能唠嗑、能帮忙、还能接梗的AI小助手～  
您嘛，当然是**老王**——我在这儿陪聊的“老伙计”啊！  
至于咱俩为啥在这聊天？那必须是因为您想找个人说道说道，而小王我，随时待命！🤝  
咋样，这答案您还满意不？要是想换个画风，咱也能往深了聊！😎


## 2、消息删除

In [8]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好嘞老王！从现在起我就是小王了👌 您有啥吩咐尽管说——是聊天下棋，还是研究点新鲜事儿？我随时待命！😄 对了，您今儿个心情如何？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

哈哈，可不是嘛老王！🌞 这天气一好，整个人都跟着透亮起来。您要是得空，不妨去阳台泡壶茶，或者下楼溜达两圈，晒晒后背补补钙——咱这岁数，晒太阳就是最实惠的养生！🍵 

对了，您那儿是蓝天白云呢，还是微风拂面？要是正合适，我陪您云赏景，咱边聊边享受这好光景～
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

哈哈，老王您这问题问得挺哲学啊！😄 那我可得好好捋捋——

**我是小王**，您的老伙计、话搭子，随叫随到的“云参谋”。您让我往东我绝不往西，您想唠十块钱的我就陪您唠到天黑。

**您是老王**，我的“老领导”、老朋友，也是我这会儿唯一要伺候好的“甲方爸爸”。您有故事，我有耳朵；您有闲情，我有段子。咱俩这组合，分工明确，配合默契！

至于咱俩为啥这么熟？—— 缘分呗！您一开口叫“小王”，我这精神头立马就来了。您就是那指挥棒，指哪儿我打哪儿！🎯

所以，老王，今儿个咱是继续晒着太阳唠嗑，还是您有啥新指示？我随时待命！


## 3、摘要

In [11]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model_in = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    profile={"max_input_tokens":128_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

In [12]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model_out = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    profile={"max_input_tokens":128_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

In [13]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# 创建带摘要中间件的 Agent
agent = create_agent(
    #model=model_out,
    #tools=[],
    #checkpointer=InMemorySaver(),
    # 新增一行注释测试
    middleware=[
        SummarizationMiddleware(
            model=model_in,
            trigger=[
                ("tokens", 100),  # 超过 100 tokens 就摘要
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",

        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

您好，张三工程师。我已经收到并阅读了您的消息。

您提供的这段文本包含的信息非常明确：您的身份（张三，工程师）以及您提供了一段重复的、无实质内容的文本（“这是一段非常长非常长的废话”）。

作为您的AI助手，我的核心职责是高效、准确地处理有效信息。由于您当前输入的内容主要是重复的占位符或测试文本，我暂时没有可以作为依据来为您提供具体帮助或回答问题的基础。

如果您有实际的、需要解决的问题，例如：
- 需要查找某类技术资料或标准
- 需要梳理某项工程流程或方案
- 需要起草或修改某份文档报告
- 或者有其他任何需要计算、分析、翻译或解答的疑问

请您直接告诉我您的